# Auditoria de factualidade apos reescrita

Este notebook e para o seu caso: voce so tem noticias reais.

Em vez de treinar fake vs real, ele mede se a reescrita contradiz o texto original (potencialmente tornou-se falsa).

## 1) Imports

In [ ]:
from pathlib import Path

import pandas as pd
import torch
from torch.nn.functional import softmax
from transformers import AutoTokenizer, AutoModelForSequenceClassification

## 2) Configuracao

Ajuste os caminhos e nomes de colunas abaixo.

In [ ]:
# Arquivo com pares: original + reescrita
PAIRED_FILE = Path("../data/science_rewritten_pairs.csv")

# Colunas dos textos
ORIGINAL_COLUMN = "description"
REWRITTEN_COLUMN = "rewritten_news"
ROW_ID_COLUMN = None  # opcional, ex.: "id"

# Modelo NLI para detectar contradicao
NLI_MODEL = "MoritzLaurer/mDeBERTa-v3-base-mnli-xnli"

# Limites para marcar potencialmente falsa apos reescrita
CONTRADICTION_THRESHOLD = 0.60
DELTA_THRESHOLD = 0.20  # contradicao - entailment

MAX_ROWS = 300
OUTPUT_FILE = Path("../data/rewriting_consistency_audit.csv")

OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
print("Configuracao carregada.")

## 3) Funcoes utilitarias

In [ ]:
def read_dataset(file_path: Path) -> pd.DataFrame:
    suffix = file_path.suffix.lower()
    if suffix == ".csv":
        return pd.read_csv(file_path)
    if suffix == ".json":
        return pd.read_json(file_path)
    raise ValueError(f"Formato nao suportado: {file_path}")

def validate_pair_columns(df: pd.DataFrame, original_col: str, rewritten_col: str) -> None:
    if original_col not in df.columns:
        raise ValueError(f"Coluna original nao encontrada: {original_col}")
    if rewritten_col not in df.columns:
        raise ValueError(f"Coluna reescrita nao encontrada: {rewritten_col}")

def consistency_flag(entailment: float, contradiction: float) -> str:
    delta = contradiction - entailment
    if contradiction >= CONTRADICTION_THRESHOLD and delta >= DELTA_THRESHOLD:
        return "potencialmente_falsa_apos_reescrita"
    return "consistente_com_original"

## 4) Carregar dados pareados

In [ ]:
paired_df = read_dataset(PAIRED_FILE).copy()
validate_pair_columns(paired_df, ORIGINAL_COLUMN, REWRITTEN_COLUMN)

paired_df[ORIGINAL_COLUMN] = paired_df[ORIGINAL_COLUMN].fillna("").astype(str).str.strip()
paired_df[REWRITTEN_COLUMN] = paired_df[REWRITTEN_COLUMN].fillna("").astype(str).str.strip()
paired_df = paired_df[(paired_df[ORIGINAL_COLUMN] != "") & (paired_df[REWRITTEN_COLUMN] != "")].copy()
paired_df = paired_df.head(MAX_ROWS).copy()

print(f"Total de pares para auditoria: {len(paired_df)}")
paired_df[[ORIGINAL_COLUMN, REWRITTEN_COLUMN]].head()

## 5) Carregar modelo NLI e avaliar contradicao

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(NLI_MODEL)
model = AutoModelForSequenceClassification.from_pretrained(NLI_MODEL)
model.eval()

id2label = {int(k): v.lower() for k, v in model.config.id2label.items()}

def nli_pair_scores(premise: str, hypothesis: str) -> dict[str, float]:
    encoded = tokenizer(
        premise,
        hypothesis,
        return_tensors="pt",
        truncation=True,
        max_length=512,
    )
    with torch.no_grad():
        logits = model(**encoded).logits[0]
    probs = softmax(logits, dim=-1).cpu().numpy()

    label_probs = {id2label[i]: float(probs[i]) for i in range(len(probs))}

    entailment = label_probs.get("entailment", label_probs.get("entails", 0.0))
    contradiction = label_probs.get("contradiction", label_probs.get("contradicts", 0.0))
    neutral = label_probs.get("neutral", 0.0)

    return {
        "entailment": entailment,
        "contradiction": contradiction,
        "neutral": neutral,
    }

rows = []
for idx, row in paired_df.iterrows():
    original_text = row[ORIGINAL_COLUMN]
    rewritten_text = row[REWRITTEN_COLUMN]

    scores = nli_pair_scores(original_text, rewritten_text)
    flag = consistency_flag(scores["entailment"], scores["contradiction"])

    out = {
        "row_index": int(idx),
        "entailment": scores["entailment"],
        "contradiction": scores["contradiction"],
        "neutral": scores["neutral"],
        "consistency_flag": flag,
    }

    if ROW_ID_COLUMN and ROW_ID_COLUMN in paired_df.columns:
        out["row_id"] = row[ROW_ID_COLUMN]

    rows.append(out)

result_df = pd.DataFrame(rows)
audit_df = paired_df.join(result_df.set_index("row_index"), how="left")

print("Resumo da auditoria:")
print(audit_df["consistency_flag"].value_counts(dropna=False))

audit_df[[ORIGINAL_COLUMN, REWRITTEN_COLUMN, "entailment", "contradiction", "consistency_flag"]].head(10)

## 6) Salvar e inspecionar casos mais criticos

In [ ]:
audit_df.to_csv(OUTPUT_FILE, index=False)
print(f"Auditoria salva em: {OUTPUT_FILE}")

suspects = audit_df.sort_values("contradiction", ascending=False).head(20)
suspects[[ORIGINAL_COLUMN, REWRITTEN_COLUMN, "entailment", "contradiction", "consistency_flag"]]